# Notebook 2: Embeddings & Semantic Similarity
### How meaning becomes a vector — built from scratch with NumPy

**No external models needed.** Uses NumPy + Python only.

---
Topics covered:
1. What is an embedding vector?
2. Cosine similarity — the core distance metric
3. Hand-crafted embeddings to build intuition
4. TF-IDF embeddings (classical, no model needed)
5. Why embedding model choice matters
6. Semantic search demo over banking documents
7. The danger of mixing embedding models

In [ ]:
import numpy as np
import math
from collections import Counter, defaultdict
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
print("All imports OK — no external models needed ✓")

## 1. What is an Embedding?
An embedding converts text into a dense numerical vector. Similar meanings → vectors that point in similar directions.

In [ ]:
# Build intuition with HAND-CRAFTED 4-dimensional embeddings
# Dimensions: [financial, risk, customer, technology]

HAND_CRAFTED = {
    # Format: [financial, risk, customer, technology]
    "loan default":           np.array([0.9, 0.9, 0.3, 0.1]),
    "credit delinquency":     np.array([0.8, 0.9, 0.4, 0.1]),
    "NPA classification":     np.array([0.9, 0.8, 0.2, 0.1]),
    "customer complaint":     np.array([0.3, 0.2, 0.9, 0.1]),
    "KYC verification":       np.array([0.4, 0.5, 0.8, 0.3]),
    "machine learning model": np.array([0.1, 0.2, 0.1, 0.9]),
    "neural network":         np.array([0.1, 0.1, 0.1, 0.95]),
    "interest rate hike":     np.array([0.9, 0.6, 0.3, 0.1]),
    "fraud detection":        np.array([0.5, 0.9, 0.5, 0.6]),
    "branch opening hours":   np.array([0.1, 0.0, 0.8, 0.1]),
}

def cosine_similarity(a, b):
    """Dot product divided by product of magnitudes"""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Compare semantically similar and dissimilar pairs
pairs = [
    ("loan default", "credit delinquency"),       # Should be HIGH
    ("loan default", "NPA classification"),        # Should be HIGH
    ("loan default", "customer complaint"),        # Should be MEDIUM
    ("loan default", "machine learning model"),    # Should be LOW
    ("loan default", "branch opening hours"),      # Should be VERY LOW
    ("fraud detection", "machine learning model"), # Should be MEDIUM-HIGH
    ("KYC verification", "customer complaint"),    # Should be MEDIUM
]

print(f"{'Pair':<55} {'Cosine Sim':>12} {'Interpretation'}")
print("-" * 100)
for a, b in pairs:
    sim = cosine_similarity(HAND_CRAFTED[a], HAND_CRAFTED[b])
    if sim > 0.85:   interp = "🟢 Very Similar"
    elif sim > 0.65: interp = "🟡 Related"
    elif sim > 0.4:  interp = "🟠 Weakly Related"
    else:            interp = "🔴 Unrelated"
    print(f"'{a}' vs '{b}'{'':<{50 - len(a) - len(b)}} {sim:>12.4f}   {interp}")

In [ ]:
# Visualize in 2D using PCA-like projection
terms = list(HAND_CRAFTED.keys())
matrix = np.array([HAND_CRAFTED[t] for t in terms])

# Manual 2D projection (PCA-style: project onto first two principal directions)
centered = matrix - matrix.mean(axis=0)
cov = np.cov(centered.T)
eigenvalues, eigenvectors = np.linalg.eigh(cov)
# Take top 2 eigenvectors
idx = np.argsort(eigenvalues)[::-1]
top2 = eigenvectors[:, idx[:2]]
projected = centered @ top2

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['red']*3 + ['blue']*2 + ['green']*2 + ['purple']*1 + ['orange']*1 + ['gray']*1
for i, (term, proj) in enumerate(zip(terms, projected)):
    ax.scatter(proj[0], proj[1], s=100, c=colors[i], zorder=3)
    ax.annotate(term, (proj[0], proj[1]), textcoords="offset points",
                xytext=(5, 5), fontsize=9)

ax.set_title('Hand-crafted Embeddings — 2D Projection\n(Similar concepts cluster together)', fontsize=13)
ax.set_xlabel('Principal Component 1')
ax.set_ylabel('Principal Component 2')
ax.grid(True, alpha=0.3)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='red', label='Credit Risk'),
    Patch(facecolor='blue', label='Compliance'),
    Patch(facecolor='green', label='Technology'),
    Patch(facecolor='purple', label='Macro'),
    Patch(facecolor='orange', label='Security'),
]
ax.legend(handles=legend_elements, loc='lower right')
plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/embedding_clusters.png', dpi=150, bbox_inches='tight')
plt.close()
print("Cluster visualization saved.")
print("\nKey insight: Similar banking concepts cluster together in vector space.")
print("This is how semantic search works — find vectors NEAR the query vector.")

## 2. TF-IDF Embeddings — Classical Semantic Search (No Model Needed)
TF-IDF is the ancestor of modern embeddings. Shows the same concept without neural networks.

In [ ]:
class TFIDFSearchEngine:
    """A complete semantic search engine using TF-IDF. Zero dependencies beyond Python."""
    
    def __init__(self):
        self.documents = []
        self.vocab = []
        self.tfidf_matrix = None
    
    def _tokenize(self, text):
        """Simple whitespace + punctuation tokeniser"""
        import re
        text = text.lower()
        tokens = re.findall(r'\b[a-z][a-z0-9]*\b', text)
        stopwords = {'the', 'a', 'an', 'and', 'or', 'is', 'are', 'was', 'were',
                     'be', 'been', 'being', 'to', 'of', 'in', 'on', 'at', 'for',
                     'with', 'by', 'from', 'as', 'shall', 'will', 'all', 'per'}
        return [t for t in tokens if t not in stopwords and len(t) > 2]
    
    def fit(self, documents):
        self.documents = documents
        tokenized = [self._tokenize(doc) for doc in documents]
        
        # Build vocabulary
        vocab_set = set()
        for tokens in tokenized:
            vocab_set.update(tokens)
        self.vocab = sorted(vocab_set)
        vocab_idx = {w: i for i, w in enumerate(self.vocab)}
        
        # TF matrix
        tf = np.zeros((len(documents), len(self.vocab)))
        for doc_idx, tokens in enumerate(tokenized):
            counts = Counter(tokens)
            for word, count in counts.items():
                tf[doc_idx, vocab_idx[word]] = count / len(tokens)
        
        # IDF vector
        df = (tf > 0).sum(axis=0)
        idf = np.log((len(documents) + 1) / (df + 1)) + 1
        
        # TF-IDF
        self.tfidf_matrix = tf * idf
        # Normalize rows
        norms = np.linalg.norm(self.tfidf_matrix, axis=1, keepdims=True)
        norms[norms == 0] = 1
        self.tfidf_matrix = self.tfidf_matrix / norms
        self.idf = idf
        self.vocab_idx = vocab_idx
        return self
    
    def query_vector(self, query):
        tokens = self._tokenize(query)
        vec = np.zeros(len(self.vocab))
        counts = Counter(tokens)
        for word, count in counts.items():
            if word in self.vocab_idx:
                vec[self.vocab_idx[word]] = (count / len(tokens)) * self.idf[self.vocab_idx[word]]
        norm = np.linalg.norm(vec)
        return vec / norm if norm > 0 else vec
    
    def search(self, query, top_k=3):
        q_vec = self.query_vector(query)
        scores = self.tfidf_matrix @ q_vec
        top_indices = np.argsort(scores)[::-1][:top_k]
        return [(self.documents[i][:120] + "...", scores[i]) for i in top_indices]

print("TFIDFSearchEngine class defined ✓")

In [ ]:
# Banking knowledge base
BANKING_DOCS = [
    "The Fixed Obligation to Income Ratio (FOIR) measures total EMI obligations against gross monthly income. Maximum permissible FOIR for salaried borrowers is 55 percent.",
    "CIBIL score is a three-digit credit score ranging from 300 to 900. A score above 700 is generally considered acceptable for retail loan applications at most banks.",
    "Know Your Customer (KYC) verification requires submission of Aadhaar card, PAN card, and recent bank statement. Video KYC is now permitted under RBI guidelines.",
    "Non-Performing Assets (NPA) are loans where interest or principal repayment is overdue by more than 90 days. Banks must provision for NPAs as per RBI prudential norms.",
    "Anti-Money Laundering (AML) controls require transaction monitoring systems to flag suspicious patterns including structuring, smurfing, and round-tripping of funds.",
    "The Marginal Cost of Funds based Lending Rate (MCLR) is the minimum interest rate below which banks cannot lend. It is revised monthly based on RBI policy rates.",
    "Capital Adequacy Ratio (CAR) or CRAR must be maintained at minimum 11.5 percent as per RBI Basel III norms. This includes Tier 1 and Tier 2 capital components.",
    "Loan against property (LAP) requires property valuation by an empanelled valuer. Maximum Loan to Value ratio is 75 percent for residential and 65 percent for commercial property.",
    "EMI calculation uses the formula: P × r × (1+r)^n / ((1+r)^n - 1) where P is principal, r is monthly interest rate, and n is tenure in months.",
    "Priority Sector Lending (PSL) mandates that scheduled commercial banks lend at least 40 percent of Adjusted Net Bank Credit to priority sectors including agriculture and MSME.",
]

engine = TFIDFSearchEngine().fit(BANKING_DOCS)

# Test queries
queries = [
    "What is the maximum debt to income ratio for a home loan?",
    "credit score requirements for borrowing",
    "identity verification documents",
    "bad loans provisioning requirements",
    "how to calculate monthly installment",
]

for query in queries:
    print(f"\nQuery: '{query}'")
    print("-" * 80)
    results = engine.search(query, top_k=2)
    for rank, (doc, score) in enumerate(results, 1):
        print(f"  #{rank} (score={score:.4f}): {doc}")

## 3. The Semantic Gap — Why Neural Embeddings Beat TF-IDF

In [ ]:
# Demonstrate the semantic gap — where keyword search fails
print("=== The Semantic Gap Demo ===")
print("The query 'credit delinquency probability' has ZERO keyword overlap with")
print("'loan default risk' — but they mean almost the same thing.\n")

# Queries that TF-IDF will FAIL on (semantic mismatch, no keyword overlap)
hard_queries = [
    "What happens when a borrower stops paying",     # Paraphrase of NPA
    "income to debt burden ratio",                    # Paraphrase of FOIR
    "credit worthiness score",                        # Paraphrase of CIBIL
    "money laundering prevention",                    # Paraphrase of AML
]

print(f"{'Query':<45} {'Top Result Score':>16} {'Verdict'}")
print("-" * 85)
for query in hard_queries:
    results = engine.search(query, top_k=1)
    doc, score = results[0]
    verdict = "✓ Found it" if score > 0.15 else "✗ Poor match"
    print(f"{query:<45} {score:>16.4f}   {verdict}")
    print(f"  → Best match: {doc[:80]}...")
    print()

print("Neural embeddings (sentence-transformers, text-embedding-3-large)")
print("would score 0.85+ on all these — because they understand meaning, not just keywords.")
print("This is why embedding model quality matters enormously for RAG pipelines.")

## 4. Simulated Neural Embeddings — Showing the Math

In [ ]:
# Simulate what high-quality neural embeddings would look like
# Using manually crafted similarity relationships to demonstrate properties

# Banking concept similarity matrix (based on domain knowledge)
# These values approximate what real embedding models would produce
concepts = [
    "loan default",
    "credit delinquency",    
    "NPA",                   
    "FOIR",
    "debt-to-income ratio",  
    "CIBIL score",
    "credit worthiness",     
    "KYC",
    "identity verification", 
    "AML",
]

# Approximate cosine similarities from a real embedding model
# Diagonal is 1.0 (self-similarity)
sim_matrix = np.array([
#  LD    CD   NPA  FOI  DTI  CIB   CW  KYC   IV  AML
  [1.00, 0.91, 0.88, 0.45, 0.44, 0.52, 0.50, 0.25, 0.22, 0.38],  # loan default
  [0.91, 1.00, 0.85, 0.43, 0.46, 0.55, 0.52, 0.24, 0.23, 0.36],  # credit delinquency
  [0.88, 0.85, 1.00, 0.41, 0.42, 0.50, 0.48, 0.22, 0.20, 0.35],  # NPA
  [0.45, 0.43, 0.41, 1.00, 0.93, 0.55, 0.52, 0.30, 0.28, 0.25],  # FOIR
  [0.44, 0.46, 0.42, 0.93, 1.00, 0.53, 0.54, 0.29, 0.27, 0.24],  # debt-to-income
  [0.52, 0.55, 0.50, 0.55, 0.53, 1.00, 0.92, 0.38, 0.35, 0.30],  # CIBIL score
  [0.50, 0.52, 0.48, 0.52, 0.54, 0.92, 1.00, 0.36, 0.34, 0.29],  # credit worthiness
  [0.25, 0.24, 0.22, 0.30, 0.29, 0.38, 0.36, 1.00, 0.90, 0.60],  # KYC
  [0.22, 0.23, 0.20, 0.28, 0.27, 0.35, 0.34, 0.90, 1.00, 0.58],  # identity verification
  [0.38, 0.36, 0.35, 0.25, 0.24, 0.30, 0.29, 0.60, 0.58, 1.00],  # AML
])

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(sim_matrix, cmap='RdYlGn', vmin=0.2, vmax=1.0)
ax.set_xticks(range(len(concepts)))
ax.set_yticks(range(len(concepts)))
ax.set_xticklabels(concepts, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(concepts, fontsize=9)
plt.colorbar(im, ax=ax, label='Cosine Similarity')
ax.set_title('Simulated Neural Embedding Similarity Matrix\n(Banking Concepts)', fontsize=13)

# Annotate cells
for i in range(len(concepts)):
    for j in range(len(concepts)):
        ax.text(j, i, f'{sim_matrix[i,j]:.2f}', ha='center', va='center',
                fontsize=7, color='black')

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/embedding_similarity_matrix.png', dpi=150, bbox_inches='tight')
plt.close()
print("Similarity matrix saved.")

print("\nNotice:")
print("  'loan default' ↔ 'credit delinquency' = 0.91  (different words, same meaning)")
print("  'FOIR' ↔ 'debt-to-income ratio'       = 0.93  (acronym matches paraphrase)")
print("  'KYC' ↔ 'identity verification'        = 0.90  (institutional vs plain language)")
print("\nThis is why neural embeddings outperform keyword search for banking Q&A.")

## 5. The Danger of Mixing Embedding Models

In [ ]:
# Demonstrate incompatible embedding spaces
np.random.seed(42)

# Simulate two models with different learned coordinate systems
# Both encode the same concepts but in completely different spaces
def model_a_embed(concept_id):
    """Model A's embedding space — arbitrary 8D"""
    base = np.array([0.1, 0.9, 0.3, 0.7, 0.5, 0.2, 0.8, 0.4])
    variation = np.array([
        [0.8, 0.1, 0.7, 0.2, 0.6, 0.1, 0.9, 0.3],
        [0.75, 0.15, 0.65, 0.25, 0.55, 0.15, 0.85, 0.35],  # similar to above
        [0.2, 0.8, 0.1, 0.9, 0.3, 0.7, 0.2, 0.8],
    ])
    v = variation[concept_id] + np.random.normal(0, 0.02, 8)
    return v / np.linalg.norm(v)

def model_b_embed(concept_id):
    """Model B's embedding space — DIFFERENT 8D coordinate system"""
    base = np.array([0.9, 0.1, 0.7, 0.3, 0.5, 0.8, 0.2, 0.6])
    variation = np.array([
        [0.3, 0.7, 0.2, 0.8, 0.4, 0.9, 0.1, 0.5],
        [0.35, 0.65, 0.25, 0.75, 0.45, 0.85, 0.15, 0.55],  # similar to above in B's space
        [0.9, 0.1, 0.8, 0.2, 0.7, 0.3, 0.6, 0.4],
    ])
    v = variation[concept_id] + np.random.normal(0, 0.02, 8)
    return v / np.linalg.norm(v)

# Concept 0: loan default
# Concept 1: credit delinquency (should be SIMILAR to concept 0)
# Concept 2: branch hours (should be DIFFERENT)

loan_default_A = model_a_embed(0)
credit_delin_A = model_a_embed(1)
branch_hours_A = model_a_embed(2)

loan_default_B = model_b_embed(0)
credit_delin_B = model_b_embed(1)

print("=== Same Model (correct) ===")
sim_same = cosine_similarity(loan_default_A, credit_delin_A)
print(f"  'loan default' [Model A] vs 'credit delinquency' [Model A]: {sim_same:.4f} ✓ HIGH")

print("\n=== Mixed Models (DANGEROUS) ===")
sim_mixed = cosine_similarity(loan_default_A, credit_delin_B)
print(f"  'loan default' [Model A] vs 'credit delinquency' [Model B]: {sim_mixed:.4f} ✗ GARBAGE")

print("\n=== What This Means for Production ===")
print("If you index documents with text-embedding-3-large today,")
print("then switch to BGE-M3 tomorrow, your entire vector index is INVALID.")
print("Queries using Model B will not find documents embedded with Model A.")
print("\nFix: Version-control your embedding model. Never mix in the same index.")

## Summary

| Concept | Key Insight |
|---------|-------------|
| Embedding | Text → dense vector; similar meaning → nearby vectors |
| Cosine similarity | Direction matters, not magnitude |
| TF-IDF | Keyword frequency + rareness; fast but misses paraphrases |
| Neural embeddings | Understand meaning; 'FOIR' ≈ 'debt-to-income ratio' |
| Model mixing | Incompatible spaces → garbage results |
| Production rule | Lock embedding model per index; never mix |
